# Autism Screening (UCI AQ-10 Adult)

The UCI Autism Screening Adult dataset has 704 responses to the AQ-10 questionnaire plus age, gender, ethnicity, and a few other demographics. The label is the screening *outcome*, not a clinical diagnosis — important distinction. So really this is "can we predict the screening tool's own output from its inputs," which mostly tests whether ML can rederive a scoring rule.

I'm doing this on purpose as a sanity-check exercise. If my model can't recover an explicit scoring rule, something is broken in my pipeline. I expect very high accuracy and that's a feature of the dataset, not a model achievement.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

RNG = np.random.default_rng(2)
DATA_DIR = Path("../../data/autism")

In [2]:
# UCI Autism Screening Adult (id 426). The zip holds an ARFF file; csv with quotechar="'" parses its data section fine.
import csv, io, urllib.request, zipfile
arff_path = DATA_DIR / "Autism-Adult-Data.arff"
if not arff_path.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    url = "https://archive.ics.uci.edu/static/public/426/autism+screening+adult.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extract("Autism-Adult-Data.arff", DATA_DIR)

lines = arff_path.read_text().splitlines()
names = [l.split()[1] for l in lines if l.lower().startswith("@attribute")]
start = next(i for i, l in enumerate(lines) if l.lower().startswith("@data")) + 1
rows = list(csv.reader([l for l in lines[start:] if l.strip()], quotechar="'", skipinitialspace=True))
df = pd.DataFrame(rows, columns=names).replace("?", np.nan)
# UCI spells two columns oddly; rename so the rest of the notebook reads cleanly
df = df.rename(columns={"jundice": "jaundice", "contry_of_res": "country_of_res"})
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())
for c in [f"A{i}_Score" for i in range(1, 11)]:
    df[c] = df[c].astype(int)
# `result` is the AQ-10 total itself, so it is left out of the features below

print(f"shape: {df.shape}  |  positive rate: {(df['Class/ASD'] == 'YES').mean():.2%}")
df.head(3)

shape: (704, 21)  |  positive rate: 26.85%


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,gender,ethnicity,jaundice,austim,country_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,1,1,0,0,1,1,0,0,...,f,White-European,no,no,United States,no,6,18 and more,Self,NO
1,1,1,0,1,0,0,0,1,0,1,...,m,Latino,no,yes,Brazil,no,5,18 and more,Self,NO
2,1,1,0,1,1,0,1,1,1,1,...,m,Latino,yes,yes,Spain,no,8,18 and more,Parent,YES


In [3]:
# normalise the label
label_col = "Class/ASD"
df[label_col] = df[label_col].astype(str).str.upper().map({"YES": 1, "NO": 0}).fillna(df[label_col])
y = df[label_col].astype(int).values

score_cols = [c for c in df.columns if c.startswith("A") and c.endswith("_Score")]
num_cols = score_cols + (["age"] if "age" in df.columns else [])
cat_cols = [c for c in ["gender", "jaundice", "ethnicity", "country_of_res"] if c in df.columns]

X_df = df[num_cols + cat_cols].copy()
for c in cat_cols:
    X_df[c] = X_df[c].fillna("unknown").astype(str)

X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
print("final feature count:", X_df.shape[1])

final feature count: 90


In [4]:
X = X_df.values.astype(float)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

for name, clf in [
    ("LogReg", Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])),
    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=2, n_jobs=2)),
]:
    auc = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc").mean()
    print(f"{name:13s}  AUC: {auc:.3f}")

# train one model on a holdout for the report
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)
final = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
print(classification_report(y_te, final.predict(X_te), digits=3))

LogReg         AUC: 0.997


RandomForest   AUC: 0.994
              precision    recall  f1-score   support

           0      0.990     1.000     0.995       103
           1      1.000     0.974     0.987        38

    accuracy                          0.993       141
   macro avg      0.995     0.987     0.991       141
weighted avg      0.993     0.993     0.993       141



## Takeaways

Cross-validated AUC is 0.997 for logistic regression and 0.994 for the random forest, and the holdout model gets 99.3% accuracy (one positive missed out of 38). That's expected. In this file the label is a deterministic function of the 10 item responses: every respondent with a total of 7 or more is labeled YES and every one with 6 or less is NO (I checked the crosstab of `result` against the label, and there are no exceptions). The model is recovering that rule, and the leftover errors come from a linear model not quite matching a hard cutoff.

The interesting question is what the model does with the *demographics*. They shouldn't add anything given the rule, and if a model leaned on them it would be worth checking whether there's bias in who got screened positive in the first place.

This is a useful exercise but I wouldn't oversell it. The dataset doesn't contain a real diagnostic ground truth, and the AQ-10 is a screening tool with known false-positive issues, particularly in adults with anxiety or ADHD. A model on top of a screener inherits all of the screener's limitations.